# NB-09 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-09**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-09.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

Read the temporal interlude first: changing a field, a limit, and the network operator are different events. This notebook replays their retained histories.

- [Volume IV, Temporal interlude T1 — Fields on a fixed graph](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#temp-t1)
- [Volume IV, Temporal interlude T4 — Known multi-event history](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#temp-t4)
- [Volume V, Chapter 7 — Events, topology states, and regimes](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c7)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-09', require_key=True)
print(setup_status)


# NB-09 — Known topology changes and event history
Companion to the Volume IV temporal interlude T1–T4. Prerequisites: NB-01 and
NB-04. We replay the existing Temporal Foundations and Event History fixtures.
No new schedule, graph, noisy observations, or hidden-state labels are created.

1. Recompute flows from the saved T1 injections on a fixed operator.
2. Resolve the saved T2 limit schedule, holding the graph fixed.
3. Reproduce T3's known outage and LODF comparison.
4. Replay stages A–H and align physical carriers before comparing exposures.

In [ ]:
NOTEBOOK_ID='NB-09'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

from methods_notebooks.session import digest
print('Locally inspectable original input declarations:',inputs['raw_inspection'])
t=SimpleNamespace(BUS_LABELS=['B1','B2','B3'],LINE_LABELS=['L12','L13','L23'])
values={}
def run_saved(name,parents=(),request=None):
    entry=inputs['requests'][name]
    values[name]=execute(name,entry['method'],request or entry['request'],parents)
    return values[name]
for name in inputs['requests']:
    if not name.startswith('physical:'):continue
    q=copy.deepcopy(inputs['requests'][name]['request']);parents=[]
    if name.startswith('physical:T3-dispatch-'):
        parent='physical:T3-response-'+name[-1];parents=[parent]
        q['inputs']['electrical_network']=copy.deepcopy(values[parent]['result']['network_operator_bundle']['context'])
    run_saved(name,parents,q)
t1=[]
physical=values['physical:T1-complete']['result']
for i,raw in enumerate(inputs['t1_input_rows']):
    row=copy.deepcopy(raw);row['operator_fingerprint']=digest(physical['network_operator_bundle'])
    row.update({f'flow_{line}_mw':physical['flow_field']['values'][i][j] for j,line in enumerate(t.LINE_LABELS)})
    row.update({f'angle_{bus}':physical['angle_field']['values'][i][j] for j,bus in enumerate(t.BUS_LABELS)})
    t1.append(row)
t2=[]
for i,raw in enumerate(inputs['t2_input_rows']):
    result=values[f'physical:T2-{i:02}']['result']['dispatch_result'];row=copy.deepcopy(raw)
    row.update(generator_b1_mw=result['generation'][0],generator_b2_mw=result['generation'][1],objective_cost=result['objective_rate'],energy_dual=result['energy_dual'],active_constraints=result['active_upper'],operator_fingerprint=digest(physical['network_operator_bundle']),ftr_B1_to_B3_per_mw=result['lmps'][2]-result['lmps'][0],ftr_B2_to_B3_per_mw=result['lmps'][2]-result['lmps'][1])
    for j,bus in enumerate(t.BUS_LABELS):row['lmp_'+bus]=result['lmps'][j]
    for j,line in enumerate(t.LINE_LABELS):
        for stem,key in [('flow_','flows'),('upper_shadow_','upper_shadow_prices'),('lower_shadow_','lower_shadow_prices')]:
            row[stem+line+('_mw' if stem=='flow_' else '')]=result[key][j]
    t2.append(row)
ops=[values[f'physical:T3-response-{i}']['result']['network_operator_bundle'] for i in range(2)]
q=copy.deepcopy(inputs['requests']['events:T3']['request'])
for key,op,declared,parent in zip(('electrical_network','outage_network'),ops,('original-T3-intact','original-T3-L12-outage'),('Temporal-Foundations-v1.0-intact','Temporal-Foundations-v1.0-L12-outage')):
    # Method-local source labels differ. Preserve both explicit labels and
    # verify every physical field before declaring the parent relationship.
    assert q['inputs'][key]['model_version']==declared and op['context']['model_version']==parent
    assert {k:v for k,v in q['inputs'][key].items() if k!='model_version'}=={k:v for k,v in op['context'].items() if k!='model_version'}
lodf=run_saved('events:T3',['physical:T3-response-0','physical:T3-response-1'],q)
assert lodf['result']['network_operator_bundle']['island_policy']=='refuse_disconnected'
assert abs(lodf['diagnostics']['certificate']['denominator'])>1e-9
resistances=[]
for i,op in enumerate(ops):
    name=f'aux:T3-resistance-{i}';q=copy.deepcopy(inputs['requests'][name]['request'])
    q['inputs']['laplacian_operator']['values']=copy.deepcopy(op['laplacian'])
    resistances.append(run_saved(name,[f'physical:T3-response-{i}'],q)['result']['resistance_distances']['values'][0])
name='aux:T3-fiedler-comparison';q=copy.deepcopy(inputs['requests'][name]['request'])
for side,op in zip(('left','right'),ops):
    q['inputs']['basis_pair'][side]['values']=np.asarray(op['basis'])[:,1:2].tolist()
    q['inputs']['basis_pair'][side]['operator_sha256']=digest(op)
fiedler=run_saved(name,['physical:T3-response-0','physical:T3-response-1'],q)['result']['basis_alignment']
ds=[values[f'physical:T3-dispatch-{i}']['result']['dispatch_result'] for i in range(2)]
outage=dict(topology_states=inputs['original_outage_labels'],operator_fingerprints=[digest(op) for op in ops],
    line_susceptance=[op['susceptance'] for op in ops],fixed_injection_mw=[100.,0.,-100.],
    direct_flows_mw=lodf['result']['flow_field']['values'][:2],lodf=lodf['result']['network_operator_bundle']['lodf'],
    lodf_predicted_outage_flows_mw=lodf['result']['flow_field']['values'][2],max_abs_lodf_reconstruction_error=lodf['diagnostics']['certificate']['direct_response_error'],
    eigenvalues=[op['eigenvalues'] for op in ops],effective_resistance_B1_B3=resistances,
    fiedler_angle_degrees=float(np.degrees(fiedler['principal_angles_radians'][0])),
    dispatch_generation_mw=[d['generation'] for d in ds],dispatch_flows_mw=[d['flows'] for d in ds],dispatch_lmps=[d['lmps'] for d in ds],
    dispatch_upper_shadow_prices=[d['upper_shadow_prices'] for d in ds],dispatch_objective=[d['objective_rate'] for d in ds],
    dispatch_active_sets=[d['active_upper'] for d in ds],ftr_B1_to_B3_per_mw=[d['lmps'][2]-d['lmps'][0] for d in ds],ftr_B2_to_B3_per_mw=[d['lmps'][2]-d['lmps'][1] for d in ds])
for name in inputs['requests']:
    if name.startswith('replay:'):run_saved(name)
assert values['replay:B-published']['result']['state_operator_history']['selected_event_ids']==['B']
assert values['replay:B-not-yet-analyst-known']['result']['state_operator_history']['selected_event_ids']==[]
full='replay:analyst_known_time:H';replayed=values[full]['result']['state_operator_history']
assert [r['stage_id'] for r in replayed['history']]==list('ABCDEFGH')
states={r['stage_id']:r for r in replayed['history']}
for stage,row in states.items():
    name='events:'+stage;q=copy.deepcopy(inputs['requests'][name]['request']);state=row['full_state_assertion'];net=q['inputs']['electrical_network']
    assert net['node_ids']==state['buses'] and net['edge_ids']==[x['line_id'] for x in state['lines']]
    assert net['susceptance']==[x['susceptance'] for x in state['lines']]
    assert q['inputs']['generation_offers']['generator_ids']==[x['generator_id'] for x in state['generators']]
    run_saved(name,[full],q)
for stage,row in states.items():
    name='paths:'+stage;q=copy.deepcopy(inputs['requests'][name]['request']);op=row['operator']
    q['inputs']['shift_factor_operator'].update(node_ids=op['node_ids'],constraint_ids=op['edge_ids'],values=op['shift_factors'])
    assert q['inputs']['balanced_path_matrix']['node_ids']==op['node_ids']
    run_saved(name,[full],q)
aligned=[]
for before,after in zip('ABCDEFG','BCDEFGH'):
    name='alignment:'+before+'→'+after;q=copy.deepcopy(inputs['requests'][name]['request'])
    for side,stage in (('left',before),('right',after)):
        op=states[stage]['operator']
        q['inputs']['basis_pair'][side].update({k:copy.deepcopy(op[k]) for k in ('node_ids','laplacian','eigenvalues','basis','gauge')})
    result=run_saved(name,[full],q)['result']['basis_alignment']
    left,right=[values['paths:'+s]['result']['exposure_matrix'] for s in (before,after)]
    common=sorted(set(left['constraint_ids'])&set(right['constraint_ids']))
    delta=[right['values'][right['constraint_ids'].index(k)][0]-left['values'][left['constraint_ids'].index(k)][0] for k in common]
    aligned.append(dict(transition=before+'→'+after,common_buses=result['common_node_ids'],common_lines=common,
        added_lines=sorted(set(right['constraint_ids'])-set(left['constraint_ids'])),removed_lines=sorted(set(left['constraint_ids'])-set(right['constraint_ids'])),
        low_span_angles_degrees=result['principal_angles_degrees'],common_bus_span_ranks=result['common_bus_span_ranks'],common_row_exposure_change_norm=float(np.linalg.norm(delta))))
q=copy.deepcopy(inputs['requests']['aux:A-sign-order-control']['request']);op=states['A']['operator'];low=np.asarray(op['basis'])[:,1:3]
for side,basis in (('left',low),('right',-low[:,::-1])):
    q['inputs']['basis_pair'][side].update(values=basis.tolist(),operator_sha256=digest(op))
invariance=run_saved('aux:A-sign-order-control',[full],q)['result']['basis_alignment']
assert max(invariance['principal_angles_radians'])<1e-10
print('Complete original support:',len(t1),len(t2),'rows; eight stages and eighteen separate clock cases.')

# Presentation assembles named result fields and declared metadata; no private
# optimizer, event inference, eigensolver or basis preparation runs locally.
history=copy.deepcopy(inputs['history_metadata']);stages=[]
label_maps={key:{} for key in ('operator','topology','basis','market_representation','constraint_set')}
for stage,known in states.items():
    state=known['full_state_assertion'];op=known['operator'];d=values['events:'+stage]['result']['dispatch_result']
    row={k:copy.deepcopy(state[k]) for k in ('stage_id','title','event_subtype','event_classes','effective_time','physical_effective_time','market_model_effective_time','publication_time','analyst_known_time','narrative','buses')}
    prices=dict(zip(state['buses'],d['nodal_lmps']));market=state['market_representation']
    published=dict(zip(market['output_labels'],(np.asarray(market['mapping_rows'])@np.asarray(d['nodal_lmps'])).tolist()))
    row.update(lines=d['edge_ids'],generators=d['generator_ids'],constraints=d['extra_constraint_ids'],market_representation_id=market['representation_id'],market_output_labels=market['output_labels'],market_mapping=market['mapping_rows'],
        demand_mw={bus:float(state['demand_mw'].get(bus,0)) for bus in state['buses']},generation_mw=dict(zip(d['generator_ids'],d['generation_mw'])),injections_mw=dict(zip(d['node_ids'],d['injections_mw'])),line_flows_mw=dict(zip(d['edge_ids'],d['flows_mw'])),
        line_shadow_prices=dict(zip(d['edge_ids'],(np.asarray(d['upper_line_shadow_prices'])-d['lower_line_shadow_prices']).tolist())),extra_shadow_prices=dict(zip(d['extra_constraint_ids'],d['extra_shadow_prices'])),nodal_lmps=prices,published_prices=published,objective_cost=d['objective_rate'],active_constraints=d['active_constraints'],
        eigenvalues=op['eigenvalues'],algebraic_connectivity=op['eigenvalues'][1],operator_shapes={k:list(np.asarray(v).shape) for k,v in [('incidence',op['incidence']),('ptdf',op['shift_factors']),('basis',op['basis']),('market_mapping',market['mapping_rows'])]},**known['original_metadata_fingerprints'])
    row['ftr_settlement_per_mw']={a+'_to_'+b:prices[b]-prices[a] for a,b in [('B1','B3'),('B1','B4'),('B2','B4'),('B5','B3')] if a in prices and b in prices}
    if 'EAST_HUB' in published:row['ftr_settlement_per_mw']['B1_to_EAST_HUB']=published['EAST_HUB']-published['B1']
    for label,field,prefix in [('operator','operator_fingerprint','O'),('topology','physical_topology_fingerprint','P'),('basis','basis_fingerprint','U'),('market_representation','market_representation_fingerprint','M'),('constraint_set','constraint_set_fingerprint','C')]:
        mapping=label_maps[label];fingerprint=row[field]
        if fingerprint not in mapping:mapping[fingerprint]=prefix+str(len(mapping)+1)
        row[label+'_label']=mapping[fingerprint]
    effects=dict.fromkeys(history['effect_keys'],False)
    if stages:
        prior=stages[-1];old=states[prior['stage_id']]['full_state_assertion']
        old_lines={x['line_id']:x for x in old['lines']};new_lines={x['line_id']:x for x in state['lines']};common_lines=set(old_lines)&set(new_lines)
        old_gens={x['generator_id']:x for x in old['generators']};new_gens={x['generator_id']:x for x in state['generators']}
        effects.update(bus_set=prior['buses']!=row['buses'],line_set=prior['lines']!=row['lines'],line_weights=any(abs(old_lines[k]['susceptance']-new_lines[k]['susceptance'])>1e-12 for k in common_lines),line_limits=any(abs(old_lines[k]['limit_mw']-new_lines[k]['limit_mw'])>1e-12 for k in common_lines),generator_set=prior['generators']!=row['generators'],generator_parameters=any(old_gens[k]!=new_gens[k] for k in set(old_gens)&set(new_gens)),constraint_set=prior['constraints']!=row['constraints'],
            operator_family=prior['operator_fingerprint']!=row['operator_fingerprint'],spectral_basis=prior['basis_fingerprint']!=row['basis_fingerprint'],market_representation=prior['market_representation_fingerprint']!=row['market_representation_fingerprint'],physical_topology=prior['physical_topology_fingerprint']!=row['physical_topology_fingerprint'],feasible_region=any(prior[k]!=row[k] for k in ('line_limit_fingerprint','generator_set_fingerprint','constraint_set_fingerprint')),dispatch_or_price_state=abs(prior['objective_cost']-row['objective_cost'])>1e-8 or prior['active_constraints']!=row['active_constraints'] or prior['nodal_lmps']!=row['nodal_lmps'])
    row['effects_from_previous_stage']=effects;stages.append(row)
history.update(stages=stages,effect_matrix=[[int(s['effects_from_previous_stage'][k]) for k in history['effect_keys']] for s in stages],unique_counts={name:len(label_maps[label]) for name,label in [('physical_topologies','topology'),('operator_families','operator'),('spectral_bases','basis'),('market_representations','market_representation'),('constraint_sets','constraint_set')]})

## T1 and T2: changing fields versus changing feasible regions
T1's injections vary while A, W, L, Ψ and the spectral basis stay fixed.
T2 holds demand at 100 MW and tightens L13 from 100 to 65 MW for hours 8–15.
Its operator is unchanged, but dispatch, duals, prices and path payoffs change.
A raw floating-point fingerprint may differ across solver/platform versions;
reproduction is checked numerically, and within-run equality labels are used
only to distinguish unchanged versus changed objects.

In [ ]:
assert len({r['operator_fingerprint'] for r in t1})==1
assert len({r['operator_fingerprint'] for r in t2})==1
hours=np.array([int(r['hour']) for r in t2])
limits=np.array([float(r['limit_L13_mw']) for r in t2])
spread=np.array([float(r['ftr_B1_to_B3_per_mw']) for r in t2])
fig,axes=setup_plot().subplots(1,2,figsize=(12,4));plt=setup_plot()
for line,color in zip(t.LINE_LABELS,['#88C0D0','#B48EAD','#A3BE8C']):
    axes[0].plot(range(24),[float(r[f'flow_{line}_mw']) for r in t1],label=line,color=color)
axes[0].set(xlabel='Saved hour',ylabel='MW',title='T1 · fields move on a fixed operator');axes[0].legend()
axes[1].step(hours,limits,where='mid',label='L13 limit (MW)',color='#88C0D0')
axes[1].step(hours,spread,where='mid',label='B1→B3 payoff (currency/MWh)',color='#D08770')
axes[1].set(xlabel='Saved hour',title='T2 · limit changes, graph does not');axes[1].legend();fig.tight_layout();plt.show()

## T3: known outage changes the operator
The L12 outage keeps a persistent row address with a zero line weight. Its zero
flow is a specified out-of-service state, not a fill value for an unknown row.
LODF predicts the new flows for the unchanged injection; dispatch is then
reoptimized separately. The intact flow's PTDF self-transfer denominator must
be nonzero. This notebook rejects disconnected operators instead of applying
a pseudoinverse as an implicit island-balancing policy.

In [ ]:
np.testing.assert_allclose(outage['direct_flows_mw'][1],outage['lodf_predicted_outage_flows_mw'],atol=1e-9)
assert lodf['result']['network_operator_bundle']['island_policy']=='refuse_disconnected'
assert abs(lodf['diagnostics']['certificate']['denominator'])>1e-9
print('Fixed-injection flows:',outage['direct_flows_mw'])
print('Reoptimized dispatch flows:',outage['dispatch_flows_mw'])
print('Reoptimized nodal prices:',outage['dispatch_lmps'])
fig,axes=plt.subplots(1,2,figsize=(12,4));x=np.arange(3)
for i,(label,color) in enumerate([('Intact','#88C0D0'),('L12 out','#D08770')]):
    axes[0].bar(x+(i-.5)*.35,outage['direct_flows_mw'][i],.35,label=label,color=color)
    axes[1].bar(x+(i-.5)*.35,outage['dispatch_lmps'][i],.35,label=label,color=color)
axes[0].set_xticks(x,t.LINE_LABELS);axes[0].set(ylabel='MW',title='Same injection · different routing')
axes[1].set_xticks(x,t.BUS_LABELS);axes[1].set(ylabel='Toy currency/MWh',title='Reoptimized dispatch prices');axes[1].legend();fig.tight_layout();plt.show()

## T4: replay the existing A–H event sequence
A base; B L13 outage; C restoration; D L14 addition; E generation added at B3;
F bus B5 and its interconnection; G EAST_HUB market remap; H nonthermal export
constraint. The effect matrix distinguishes topology, operators, generation,
constraints and representation. An unchanged graph does not imply unchanged
prices. G's mapping changes published coordinates without changing dispatch.
Physical effective time, market-model time, publication and analyst knowledge
time remain separate recorded fields. This is a known-event replay, not a filter.

In [ ]:
stages=history['stages']
for s in stages:
    print(s['stage_id'],s['title'],{'operator':s['operator_label'],'market':s['market_representation_label'],
        'effective':s['effective_time'],'physical':s['physical_effective_time'],
        'market_model':s['market_model_effective_time'],'published':s['publication_time'],'known':s['analyst_known_time']})
assert stages[0]['operator_label']==stages[2]['operator_label']
assert stages[3]['operator_label']==stages[4]['operator_label']
assert stages[5]['nodal_lmps']==stages[6]['nodal_lmps']
fig,ax=plt.subplots(figsize=(13,4))
from matplotlib.colors import ListedColormap
ax.imshow(history['effect_matrix'],aspect='auto',cmap=ListedColormap(['#ECEFF4','#5E81AC']),vmin=0,vmax=1)
ax.set_yticks(range(8),[s['stage_id'] for s in stages]);ax.set_xticks(range(len(history['effect_keys'])),[k.replace('_',' ') for k in history['effect_keys']],rotation=45,ha='right')
ax.set(title='Change from preceding stage · blue = changed, pale = unchanged',ylabel='Existing event stage');fig.tight_layout();plt.show()

## Compare subspaces on matched carriers
Restrict each low two-mode basis to common bus IDs, then build rank-revealing
orthonormal bases before principal-angle comparison. At E→F the bus set grows;
this common-bus comparison is conditional and does not measure the unobserved
new coordinate. It is not a claim that individual eigenvectors retain identity.
Within-span sign flips and rotations leave the comparison unchanged. A cluster
crossing the chosen two-mode boundary would require a broader spectral band.

For the balanced B1→B3 transfer, compare only common named line rows and list
additions/removals separately. Missing rows never become zero sensitivities.
The derivative for this transfer can remain almost unchanged even while a
common-bus low-mode span changes substantially.

In [ ]:
for row in aligned:print(row)
assert max(invariance['principal_angles_radians'])<1e-10
fig,axes=plt.subplots(1,2,figsize=(12,4));labels=[r['transition'] for r in aligned]
axes[0].bar(range(7),[max(r['low_span_angles_degrees']) for r in aligned],color='#B48EAD')
axes[0].set_xticks(range(7),labels);axes[0].set(ylabel='Largest principal angle, degrees',title='Low-mode spans · common bus coordinates')
axes[1].bar(range(7),[r['common_row_exposure_change_norm'] for r in aligned],color='#88C0D0')
axes[1].set_xticks(range(7),labels);axes[1].set(ylabel='Euclidean norm, MW/MW',title='B1→B3 exposure · common line rows');fig.tight_layout();plt.show()

## Exercise and limits
Classify E, G and H without looking at prices: generator set, representation,
and constraint set change respectively. Their physical graphs remain fixed.
Therefore a price discontinuity alone cannot certify a topology event, in this
fixture or in actual PJM prices. Real event attribution needs an observed model
version and an admissible knowledge-time record. Islanded systems require
explicit per-island balancing and feasibility checks, outside these fixtures.

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
checked=dict(history=history,aligned_comparisons=aligned,outage=outage,t1=t1,t2=t2,
    clock_cases={k:v['result']['state_operator_history'] for k,v in values.items() if k.startswith('replay:')},
    source_claim_strings_are_unaccepted_narrative_only=True,scientific_acceptance=False,new_synthetic_data=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(saved_T1_T2_parity=True,LODF_recomputation=True,event_history_parity=True,balanced_transfer_identity=True,common_carrier_alignment=True,connected_operator_gate=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)